# Flight Delay Analysis
**Dataset:** 2015 US domestic flights (~5.8M rows)  
**Goal:** Predict arrival/departure delays

## 1. Setup & Imports

In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('..')

## 2. Data Loading

In [101]:
# Explicit dtypes cut memory usage roughly in half
FLIGHT_DTYPES = {
    # 'YEAR': 'int16', # alles 2015 -> raus
    
    'MONTH': 'int8', 
    'DAY': 'int8', 
    'DAY_OF_WEEK': 'int8',

    'AIRLINE': 'string',

    'FLIGHT_NUMBER': 'int32', 
    
    # 'TAIL_NUMBER' viele missing, irrelevant -> raus
    'ORIGIN_AIRPORT': 'string',
    'DESTINATION_AIRPORT': 'string',

    'SCHEDULED_DEPARTURE': 'string',  
    'DEPARTURE_TIME': 'string', # 1% missing -> raus
    'DEPARTURE_DELAY': 'float32', # 1% missing = DEPARTURE_TIME

    #'TAXI_OUT': 'float32', # 2% viele missing -> raus
    #'WHEELS_OFF': 'string', # 2% viele missing = TAXI_OFF

    'SCHEDULED_TIME': 'float32', # 6 mising
    'ELAPSED_TIME': 'float32', # 2% viele missing
    #'AIR_TIME': 'float32', # 2% viele missing = ELAPSED
    'DISTANCE': 'int32',
    
    #'WHEELS_ON': 'string',  # # 2% viele missing -> raus
    #'TAXI_ON': 'float32', # 2% viele missing = WHEELS_ON

    'SCHEDULED_ARRIVAL' : 'string', 
    'ARRIVAL_TIME': 'string',
    'ARRIVAL_DELAY': 'float32', 

    #'DIVERTED': 'bool', # alle 0 -> raus
    #'CANCELLED': 'bool', # 99.99% sind 0 -> raus

    #'AIRSYSTEM_DELAY'    _> raus
    #'SECURITY_DELAY'     _> raus
    #'AIRLINE_DELAY'      _> raus
    #'LATEAIRCRAFT_DELAY' _> raus
    #'WEATHER_DELAY'      _> raus
}

print('Loading flights.csv (~592 MB) ...')
flights = pd.read_csv(DATA_DIR / 'flights.csv', dtype=FLIGHT_DTYPES)
airlines = pd.read_csv(DATA_DIR / 'airlines.csv')
airports = pd.read_csv(DATA_DIR / 'airports.csv')

# Keep only columns defined in FLIGHT_DTYPES
flights = flights[list(FLIGHT_DTYPES.keys())]
before = len(flights)

# Drop rows with any missing value
flights = flights.dropna()
after = len(flights)

print(f'dropped {before-after} rows...')
print(f'flights: {flights.shape[0]:,} rows × {flights.shape[1]} cols')

Loading flights.csv (~592 MB) ...
dropped 105071 rows...
flights: 5,714,008 rows × 16 cols


## 2a. Datenbereinigung

In [102]:
valid_airlines = set(airlines['IATA_CODE'])
before = len(flights)
invalid_airlines = flights.loc[~flights['AIRLINE'].isin(valid_airlines), 'AIRLINE'].value_counts()
flights = flights[flights['AIRLINE'].isin(valid_airlines)]
print(f'Airline-Filter: {before - len(flights):,} Zeilen entfernt → {len(flights):,} verbleibend')
print('\nEntfernte Airline IATA-Codes:')
print(invalid_airlines.to_string() if len(invalid_airlines) else '  keine')

Airline-Filter: 0 Zeilen entfernt → 5,714,008 verbleibend

Entfernte Airline IATA-Codes:
  keine


In [103]:
valid_airports = set(airports['IATA_CODE'])
before = len(flights)
invalid_origins = flights.loc[~flights['ORIGIN_AIRPORT'].isin(valid_airports), 'ORIGIN_AIRPORT'].value_counts()
invalid_dests = flights.loc[~flights['DESTINATION_AIRPORT'].isin(valid_airports), 'DESTINATION_AIRPORT'].value_counts()
flights = flights[
    flights['ORIGIN_AIRPORT'].isin(valid_airports) &
    flights['DESTINATION_AIRPORT'].isin(valid_airports)
]
print(f'Airport-Filter: {before - len(flights):,} Zeilen entfernt → {len(flights):,} verbleibend')
print('\nEntfernte ORIGIN_AIRPORT-Codes:')
print(invalid_origins.to_string() if len(invalid_origins) else '  keine')
print('\nEntfernte DESTINATION_AIRPORT-Codes:')
print(invalid_dests.to_string() if len(invalid_dests) else '  keine')

Airport-Filter: 482,878 Zeilen entfernt → 5,231,130 verbleibend

Entfernte ORIGIN_AIRPORT-Codes:
ORIGIN_AIRPORT
10397    32509
13930    27566
11298    20586
11292    18077
12892    17628
14771    14094
12266    13091
14107    12884
12889    12649
13487    10552
14747    10361
10721    10091
11433     9882
11057     9537
11618     9508
13204     9006
14869     8738
12953     8447
12478     8255
10821     7991
13232     7598
11278     6759
14679     6184
14100     6065
13303     5929
11259     5873
11697     5753
15304     5125
12191     4681
14057     4539
10693     4450
15016     4295
13796     3994
10423     3871
12173     3761
13495     3646
14893     3598
13198     3536
14908     3524
14831     3457
11042     3202
14492     2984
12264     2979
13342     2631
14683     2594
12339     2361
11066     2178
14122     2122
11193     1903
14843     1858
13830     1814
10140     1805
14635     1756
10800     1753
13891     1734
14027     1734
10529     1705
12451     1701
14524     1628
138

In [104]:
out_path = DATA_DIR / 'flights_cut.csv'
flights.to_csv(out_path, index=False)
print(f'Saved {flights.shape[0]:,} rows × {flights.shape[1]} cols → {out_path}')

Saved 5,231,130 rows × 16 cols → ../flights_cut.csv


## 2b. Train / Validation Split

In [79]:
train.head(10)

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY
1964290,5,5,2,DL,2121,ANC,SEA,1747,1801,14.0,193.0,193.0,1448,2200,2214,14.0
3677774,8,17,1,WN,1687,LAS,HOU,0535,0530,-5.0,180.0,178.0,1235,1035,1028,-7.0
781663,2,21,6,DL,1353,ATL,PNS,1100,1054,-6.0,77.0,79.0,271,1117,1113,-4.0
3609226,8,12,3,WN,736,HOU,MSY,1910,1922,12.0,60.0,59.0,302,2010,2021,11.0
5420722,12,6,7,OO,6488,MSP,IAH,1045,1048,3.0,173.0,165.0,1034,1338,1333,-5.0
3411820,8,1,6,AS,733,BOS,SEA,0700,0703,3.0,369.0,370.0,2496,1009,1013,4.0
1542165,4,9,4,MQ,3368,JAN,DFW,1357,1421,24.0,95.0,92.0,408,1532,1553,21.0
3127341,7,15,3,OO,6197,BUR,SFO,1100,1103,3.0,79.0,92.0,326,1219,1235,16.0
2176335,5,18,1,EV,5176,DTW,OMA,1945,1946,1.0,125.0,126.0,651,2050,2052,2.0
5123689,11,16,1,B6,1680,FLL,DCA,2020,2026,6.0,142.0,143.0,899,2242,2249,7.0


## 3. Feature Engineering

In [105]:
# Encoder-Maps: IATA-Code → int
airline_codes = sorted(flights['AIRLINE'].unique())
airport_codes = sorted(set(flights['ORIGIN_AIRPORT'].unique()) | set(flights['DESTINATION_AIRPORT'].unique()))
airline_map = {c: i for i, c in enumerate(airline_codes)}
airport_map = {c: i for i, c in enumerate(airport_codes)}

# Distanz-Lookup: (origin, dest) → Median aus den Flugdaten
dist_lookup = (
    flights.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])['DISTANCE']
    .median()
    .to_dict()
)
dist_mean = flights['DISTANCE'].mean()

# Feature-Matrix aufbauen
feat = flights.copy()
feat['AIRLINE_ENC'] = feat['AIRLINE'].map(airline_map)
feat['ORIGIN_ENC']  = feat['ORIGIN_AIRPORT'].map(airport_map)
feat['DEST_ENC']    = feat['DESTINATION_AIRPORT'].map(airport_map)

hour = feat['SCHEDULED_DEPARTURE'].str.zfill(4).str[:2].astype(int)
feat['HOUR'] = hour
#feat['HOUR_SIN']  = np.sin(2 * np.pi * hour / 24)
#feat['HOUR_COS']  = np.cos(2 * np.pi * hour / 24)
#feat['MONTH_SIN'] = np.sin(2 * np.pi * feat['MONTH'] / 12)
#feat['MONTH_COS'] = np.cos(2 * np.pi * feat['MONTH'] / 12)
#feat['DOW_SIN']   = np.sin(2 * np.pi * feat['DAY_OF_WEEK'] / 7)
#feat['DOW_COS']   = np.cos(2 * np.pi * feat['DAY_OF_WEEK'] / 7)

REG_FEATURES = [
    #'HOUR_SIN','HOUR_COS',
    'HOUR','MONTH','DAY_OF_WEEK',
    #'MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS',
    #'HOUR_SIN', 'HOUR_COS',
    'AIRLINE_ENC', 'ORIGIN_ENC', 'DEST_ENC', 'DISTANCE',
]
print(f'Features:  {REG_FEATURES}')
print(f'Datensatz: {len(feat):,} Zeilen')

Features:  ['HOUR', 'MONTH', 'DAY_OF_WEEK', 'AIRLINE_ENC', 'ORIGIN_ENC', 'DEST_ENC', 'DISTANCE']
Datensatz: 5,231,130 Zeilen


## 4. Modell-Training (Regression)

In [106]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

X = feat[REG_FEATURES]
y = feat['ARRIVAL_DELAY']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f'MAE:  {mean_absolute_error(y_test, preds):.1f} min')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.1f} min')

Train: 4,184,904 | Test: 1,046,226
MAE:  20.9 min
RMSE: 38.8 min


### Modell 2: Ridge Regression (linearer Baseline)

In [108]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0)),
])
ridge_pipe.fit(X_train, y_train)
ridge_preds = ridge_pipe.predict(X_test)

mae_hgb   = mean_absolute_error(y_test, preds)
rmse_hgb  = np.sqrt(mean_squared_error(y_test, preds))
mae_ridge  = mean_absolute_error(y_test, ridge_preds)
rmse_ridge = np.sqrt(mean_squared_error(y_test, ridge_preds))

print(f'{"Modell":<30} {"MAE":>8} {"RMSE":>8}')
print('-' * 48)
print(f'{"HistGradientBoosting":<30} {mae_hgb:>7.1f} {rmse_hgb:>7.1f}')
print(f'{"Ridge":<30} {mae_ridge:>7.1f} {rmse_ridge:>7.1f}')

Modell                              MAE     RMSE
------------------------------------------------
HistGradientBoosting              20.9    38.8
Ridge                             21.4    39.5


## 5. Vorhersage-Funktion

In [109]:
from datetime import datetime

def predict_delay(date: str, time: str, origin: str, dest: str, airline: str) -> float:
    """
    date:    'YYYY-MM-DD'
    time:    'HHMM'  z.B. '0830'
    origin / dest / airline: IATA-Codes
    Gibt vorhergesagte Ankunftsverspätung in Minuten zurück.
    """
    dt   = datetime.strptime(date, '%Y-%m-%d')
    hour = int(time[:2])
    dist = dist_lookup.get((origin, dest), dist_lookup.get((dest, origin), dist_mean))

    row = pd.DataFrame([{
        #'MONTH_SIN':   np.sin(2 * np.pi * dt.month / 12),
        #'MONTH_COS':   np.cos(2 * np.pi * dt.month / 12),
        #'DOW_SIN':     np.sin(2 * np.pi * dt.weekday() / 7),
        #'DOW_COS':     np.cos(2 * np.pi * dt.weekday() / 7),
        #'HOUR_SIN':    np.sin(2 * np.pi * hour / 24),
        #'HOUR_COS':    np.cos(2 * np.pi * hour / 24),
        'HOUR':    hour,
        #'SCHEDULED_DEPARTURE':0,
        'MONTH':dt.month,
        'DAY_OF_WEEK':dt.weekday(),
        'AIRLINE_ENC': airline_map.get(airline, -1),
        'ORIGIN_ENC':  airport_map.get(origin, -1),
        'DEST_ENC':    airport_map.get(dest, -1),
        'DISTANCE':    dist,
    }])
    return round(float(model.predict(row)[0]), 1)

# Beispiele
examples = [
    ('2015-07-04', '0800', 'LAX', 'JFK', 'AA'),
    ('2015-12-24', '1800', 'ORD', 'ATL', 'WN'),
    ('2015-03-15', '1200', 'SFO', 'SEA', 'AS'),
]
for args in examples:
    delay = predict_delay(*args)
    label = f'+{delay} min (verspätet)' if delay > 0 else f'{delay} min (pünktlich)'
    print(f'{args[4]}  {args[2]}→{args[3]}  {args[0]} {args[1]}  →  {label}')

AA  LAX→JFK  2015-07-04 0800  →  -3.7 min (pünktlich)
WN  ORD→ATL  2015-12-24 1800  →  +19.0 min (verspätet)
AS  SFO→SEA  2015-03-15 1200  →  +1.9 min (verspätet)
